# Model Building — Per-Zone Hourly Demand Forecasting

Continues from `Data_Analysis_Preprocessing.ipynb`. Here we build the **first forecasting model(s)** for the AutoRetrain-NYC pipeline:

- **Target**: hourly pickup demand, **per zone** (multiple series, not just the NYC-wide total)
- **Baselines**: Naive Seasonal (24h), Naive Mean, Naive Drift — fit independently per zone
- **First real model**: a single **global** LightGBM regression model (via Darts' `LightGBMModel`) trained jointly across all selected zones, using lag features + calendar covariates
- **Output**: per-zone error metrics for baseline vs. real model, comparison plots, and a saved model artifact that later pipeline stages (retrain / evaluate / promote) can consume

A single **global** model (rather than one model per zone) is the right first step for this project: it's what the closed-loop retraining/promotion pipeline described in the project overview will retrain and re-evaluate going forward, and it scales to all zones without maintaining hundreds of separate models.

In [7]:
!pip install -q darts lightgbm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.9/67.9 kB 846.7 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 841.3/841.3 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 17.7 MB/s eta 0:00:00


## Imports

In [8]:
import os
import gc
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

from darts import TimeSeries
from darts.models import NaiveSeasonal, NaiveMean, NaiveDrift, LightGBMModel
from darts.metrics import mae, rmse, mape
from darts.dataprocessing.transformers import Scaler

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (14, 5)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

COLORS = {
    "blue": "#2563EB",
    "purple": "#7C3AED",
    "green": "#059669",
    "amber": "#F59E0B",
    "red": "#DC2626",
    "slate": "#475569",
    "light_blue": "#60A5FA",
}

## Paths

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [11]:
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/AutoRetrain-NYC")

PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
FEATURE_DIR = PROJECT_DIR / "data" / "features"
MODEL_DIR = PROJECT_DIR / "models"
METRICS_DIR = PROJECT_DIR / "metrics"

for d in (FEATURE_DIR, MODEL_DIR, METRICS_DIR):
    d.mkdir(parents=True, exist_ok=True)

HOURLY_ZONE_PATH = PROCESSED_DIR / "nyc_hourly_demand_gapfilled_2026.parquet"

print("Loading:", HOURLY_ZONE_PATH)

Loading: /content/drive/MyDrive/AutoRetrain-NYC/data/processed/nyc_hourly_demand_gapfilled_2026.parquet


## Load the processed zone-level data
This is the `(timestamp, PULocationID, demand)` table produced by the preprocessing notebook — hourly pickup counts per zone, before any reindexing or gap-filling.

In [13]:
hourly_zone_demand = pd.read_parquet(HOURLY_ZONE_PATH)
hourly_zone_demand["timestamp"] = pd.to_datetime(hourly_zone_demand["timestamp"])

# print("Shape:", hourly_zone_demand.shape)
# print("Zones:", hourly_zone_demand["PULocationID"].nunique())
# print("Range:", hourly_zone_demand["timestamp"].min(), "→", hourly_zone_demand["timestamp"].max())

hourly_zone_demand.head()

,timestamp,demand,was_missing
0,2026-01-01 00:00:00,9294.0,False
1,2026-01-01 01:00:00,10254.0,False
2,2026-01-01 02:00:00,7853.0,False
3,2026-01-01 03:00:00,6529.0,False
4,2026-01-01 04:00:00,4108.0,False


## Select zones to model
Modeling all ~260 TLC zones means dragging along a long tail of near-empty series that add noise more than signal for a first model. We reuse the concentration check from the EDA notebook and keep the smallest set of zones needed to cover **95% of total demand** — the rest can be added back (or bucketed as `"other"`) once the pipeline is proven out.

In [ ]:
zone_totals = (
    hourly_zone_demand
    .groupby("PULocationID")["demand"]
    .sum()
    .sort_values(ascending=False)
)

zone_share = zone_totals / zone_totals.sum()
zone_cumshare = zone_share.cumsum()

COVERAGE_TARGET = 0.95
SELECTED_ZONES = zone_cumshare[zone_cumshare <= COVERAGE_TARGET].index.tolist()
# always include the zone that crosses the threshold
if len(SELECTED_ZONES) < len(zone_cumshare):
    SELECTED_ZONES.append(zone_cumshare.index[len(SELECTED_ZONES)])

print(f"Zones selected: {len(SELECTED_ZONES)} / {len(zone_totals)}")
print(f"Coverage achieved: {zone_share.loc[SELECTED_ZONES].sum() * 100:.1f}% of total demand")

## Build a complete hourly panel per zone
Two different kinds of "missing" need two different fixes:

- **Timestamps missing from the whole dataset** (system-wide gaps, already identified in the EDA notebook) → these are true data gaps and should be **interpolated**, same as we did for the NYC-wide series.
- **A given zone simply has no rows for an hour that other zones do have** → that's a **genuine zero** (no pickups), not a gap, and should be filled with `0`.

Getting this distinction right matters — filling real zero-demand hours with an interpolated value would systematically inflate quiet zones and quiet hours.

In [ ]:
panel = (
    hourly_zone_demand[hourly_zone_demand["PULocationID"].isin(SELECTED_ZONES)]
    .pivot(index="timestamp", columns="PULocationID", values="demand")
)

expected_hours = pd.date_range(
    start=hourly_zone_demand["timestamp"].min(),
    end=hourly_zone_demand["timestamp"].max(),
    freq="h",
)

# Hours that exist somewhere in the raw data (i.e. not a system-wide gap)
observed_hours = pd.DatetimeIndex(hourly_zone_demand["timestamp"].unique())
system_missing_hours = expected_hours.difference(observed_hours)

panel = panel.reindex(expected_hours)
panel.index.name = "timestamp"

# Zero-fill genuine "zone had no trips this hour" gaps
panel.loc[panel.index.difference(system_missing_hours)] = (
    panel.loc[panel.index.difference(system_missing_hours)].fillna(0.0)
)

# Interpolate genuine system-wide gaps
panel = panel.interpolate(method="linear")

print("Panel shape (hours x zones):", panel.shape)
print("System-wide missing hours (interpolated):", len(system_missing_hours))
print("Remaining NaNs after fill:", panel.isna().sum().sum())

panel.head()

## Convert to Darts TimeSeries (one per zone) + calendar covariates
We keep this as a **list of univariate series** (rather than one multivariate series) — that's the format Darts' global models expect for multi-series training, and it matches how the pipeline will likely add/drop zones over time.

In [ ]:
zone_series = TimeSeries.from_times_and_values(
    times=panel.index,
    values=panel.values,
    columns=[str(c) for c in panel.columns],
    freq="h",
)

series_list = [zone_series.univariate_component(i) for i in range(zone_series.n_components)]
zone_ids = [str(c) for c in panel.columns]

print(f"Built {len(series_list)} per-zone series, each of length {len(series_list[0])}")

# Calendar covariates, shared across all zones (known in advance -> valid as future covariates)
calendar_df = pd.DataFrame(index=panel.index)
calendar_df["hour"] = calendar_df.index.hour
calendar_df["day_of_week"] = calendar_df.index.dayofweek
calendar_df["is_weekend"] = calendar_df["day_of_week"].isin([5, 6]).astype(int)
calendar_df["day_of_month"] = calendar_df.index.day
calendar_df["month"] = calendar_df.index.month

calendar_covariates = TimeSeries.from_times_and_values(
    times=calendar_df.index,
    values=calendar_df.values,
    columns=calendar_df.columns.tolist(),
    freq="h",
)

covariates_list = [calendar_covariates] * len(series_list)

## Train / test split
Time-based split, held out at the *end* of the series for every zone (no shuffling — this is a forecasting problem). We hold out the **last 14 days** as the test set.

In [ ]:
TEST_HORIZON = 14 * 24  # 14 days of hourly data

train_series = [s[:-TEST_HORIZON] for s in series_list]
test_series = [s[-TEST_HORIZON:] for s in series_list]

print("Train length:", len(train_series[0]))
print("Test length:", len(test_series[0]))
print("Test window:", test_series[0].start_time(), "→", test_series[0].end_time())

## Baseline models (fit per zone)
Simple, cheap benchmarks any real model has to beat: a naive seasonal repeat of the last day, the historical mean, and a linear drift. These are fit **independently per zone** since Darts' naive models are local.

In [ ]:
def evaluate_local_baseline(model_cls, model_kwargs=None):
    model_kwargs = model_kwargs or {}
    rows = []

    for zone_id, train_s, test_s in zip(zone_ids, train_series, test_series):
        model = model_cls(**model_kwargs)
        model.fit(train_s)
        pred = model.predict(TEST_HORIZON)

        rows.append({
            "zone": zone_id,
            "mae": mae(test_s, pred),
            "rmse": rmse(test_s, pred),
            "mape": mape(test_s, pred),
        })

    return pd.DataFrame(rows)

baseline_results = {
    "naive_seasonal_24h": evaluate_local_baseline(NaiveSeasonal, {"K": 24}),
    "naive_mean": evaluate_local_baseline(NaiveMean),
    "naive_drift": evaluate_local_baseline(NaiveDrift),
}

for name, df in baseline_results.items():
    print(f"{name:>20s} | MAE {df['mae'].mean():7.2f} | RMSE {df['rmse'].mean():7.2f} | MAPE {df['mape'].mean():6.2f}%")

## First real model: global LightGBM
One `LightGBMModel` trained **jointly** across every selected zone's series, using:
- **lags**: recent history (last 24h) plus the same hour on the last 2 and last 7 days, to capture daily and weekly seasonality directly as features
- **future covariates**: the calendar features built above (known ahead of time for any forecast horizon)

Training one global model across zones — rather than one model per zone — is what makes this practical to retrain automatically as new data arrives, which is the point of the AutoRetrain pipeline.

In [ ]:
LAGS = [-1, -2, -3, -24, -25, -48, -168]

lgbm_model = LightGBMModel(
    lags=LAGS,
    lags_future_covariates=[0],
    output_chunk_length=24,
    random_state=42,
    verbose=-1,
)

lgbm_model.fit(
    series=train_series,
    future_covariates=covariates_list,
)

print("Model trained on", len(train_series), "zone series.")

In [ ]:
lgbm_predictions = lgbm_model.predict(
    n=TEST_HORIZON,
    series=train_series,
    future_covariates=covariates_list,
)

lgbm_rows = []
for zone_id, test_s, pred in zip(zone_ids, test_series, lgbm_predictions):
    lgbm_rows.append({
        "zone": zone_id,
        "mae": mae(test_s, pred),
        "rmse": rmse(test_s, pred),
        "mape": mape(test_s, pred),
    })

lgbm_results = pd.DataFrame(lgbm_rows)

print(f"{'lightgbm_global':>20s} | MAE {lgbm_results['mae'].mean():7.2f} | RMSE {lgbm_results['rmse'].mean():7.2f} | MAPE {lgbm_results['mape'].mean():6.2f}%")

## Compare baselines vs. the real model, per zone

In [ ]:
all_results = []
for name, df in baseline_results.items():
    df = df.copy()
    df["model"] = name
    all_results.append(df)

lgbm_tagged = lgbm_results.copy()
lgbm_tagged["model"] = "lightgbm_global"
all_results.append(lgbm_tagged)

comparison = pd.concat(all_results, ignore_index=True)

summary = (
    comparison
    .groupby("model")[["mae", "rmse", "mape"]]
    .mean()
    .sort_values("mae")
)

display(summary)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

sns.boxplot(
    data=comparison,
    x="model",
    y="mae",
    order=summary.index,
    palette=[COLORS["slate"], COLORS["light_blue"], COLORS["purple"], COLORS["green"]],
)

ax.set_title("Per-Zone MAE by Model", fontsize=16, fontweight="bold")
ax.set_xlabel("Model")
ax.set_ylabel("MAE (trips/hour)")

plt.tight_layout()
plt.show()

## Inspect predictions on a few sample zones
One high-volume zone and one low-volume zone, so we can sanity-check that the model is tracking both regimes reasonably.

In [ ]:
sample_zone_ids = [zone_ids[0], zone_ids[-1]]  # highest & lowest-volume of the selected zones

fig, axes = plt.subplots(len(sample_zone_ids), 1, figsize=(16, 5 * len(sample_zone_ids)))

for ax, zid in zip(np.atleast_1d(axes), sample_zone_ids):
    idx = zone_ids.index(zid)

    test_series[idx].plot(ax=ax, label="Actual", color=COLORS["slate"])
    lgbm_predictions[idx].plot(ax=ax, label="LightGBM (global)", color=COLORS["blue"])

    ax.set_title(f"Zone {zid} — Actual vs. Predicted (test window)", fontsize=14, fontweight="bold")
    ax.set_xlabel("Time")
    ax.set_ylabel("Trips per hour")
    ax.legend()

plt.tight_layout()
plt.show()

## Save the model + metrics
These artifacts are what the later pipeline stages (retrain trigger, challenger evaluation, promotion gate) will consume: the trained global model, the zone list it was trained on, and its baseline-relative performance.

In [ ]:
MODEL_PATH = MODEL_DIR / "lightgbm_global_v1.pt"
lgbm_model.save(str(MODEL_PATH))

ZONE_LIST_PATH = FEATURE_DIR / "selected_zones_v1.csv"
pd.Series(SELECTED_ZONES, name="PULocationID").to_csv(ZONE_LIST_PATH, index=False)

METRICS_PATH = METRICS_DIR / "model_v1_comparison.csv"
comparison.to_csv(METRICS_PATH, index=False)

print("Saved model to:", MODEL_PATH)
print("Saved zone list to:", ZONE_LIST_PATH)
print("Saved metrics to:", METRICS_PATH)
print()
print("Summary (mean across zones):")
print(summary)

## Takeaways & next steps
- The global LightGBM model should be compared above against the naive seasonal baseline (repeating the same hour last day) — beating that, not the mean/drift baselines, is the real bar for a demand-forecasting model with strong daily/weekly seasonality.
- If MAPE is dominated by a handful of very low-volume zones (division by small numbers), consider reporting weighted MAPE (by zone volume) or MAE/RMSE as the primary metrics instead.
- Next pipeline stages: wrap this training + evaluation code into a script/job (candidate model training), add a **quality gate** that compares a newly trained candidate against the current production model on a fixed holdout, and only promote the candidate if it wins.
- Zones excluded from `SELECTED_ZONES` (long tail, <5% of demand) can be added later, either individually or bucketed into a single `"other"` series.